In [10]:
T = CartanType(['A', 6])
W = WeylGroup(T,prefix="s")
[s1, s2, s3, s4, s5, s6] = W.simple_reflections()

WCR = WeylCharacterRing(T, style="coroots", base_ring=QQ)
R = WeightRing(WCR)
# EXPI MODIFIED, THIS CODE ONLY WORKS IN TYPE A
f = [w.coerce_to_sl() for w in WCR.fundamental_weights()]

n = len(W.simple_reflections())
e = s1^2
w0 = W.long_element()
S.<q> = LaurentPolynomialRing(QQ)
KL = KazhdanLusztigPolynomial(W,q)
q = 7

def Iw(w):
    a = []
    for i in range(1, n+1):
        if w.has_right_descent(i):
            a.append(i)
    return a

def Iwc(w):
    a = []
    for i in range(1, n+1):
        if not w.has_right_descent(i):
            #a.append(W.simple_reflections()[i])
            a.append(i)
    return a

def expI(l):
    a = R.one()
    for i in l:
        a = a * R(f[i-1])
    return a


def Cpsub_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(y)
        s += int(KL.P(y, w)(1)) * a
    return s


def Cpsup_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w0*w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(w0*y)
        s += int(KL.P(y, w0*w)(1)) * a
    return s


def fsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)))


def qfsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)).scale(q))


def fsup(w):
    return Cpsup_on_wr_element(w, expI(Iw(w)))


invols = []
for w in W:
    if w*w == e:
        invols.append(w)

def Iweight(w):
    wgt = 0
    for a in Iw(w):
        wgt += f[a-1]
    return wgt

rho = Iweight(w0)


def weightpairing(l):
    a = WCR.dot_reduce(l - rho)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])


def woke_pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

d_sub = {}
d_sup = {}

for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_sub[i] = fsub(W[i])
    d_sup[i] = fsup(W[i])

print("Setup done, f_w, f^w are defined.")

KeyboardInterrupt: 

In [2]:
def build_matrix():
    m = []
    for i in range(len(W)):
        print(i + 1, "out of", len(W), end='\r')
        r = []
        for j in range(len(W)):
            r.append(woke_pairing(d_sub[i], d_sup[j]))
        m.append(r)
    return m
bigmat = build_matrix()
#bigmat[i][j] = woke_pairing(d_sub[i], d_sup[j]) for all i, j
alls = list(range(len(W)))
pops = list(range(len(W)))
order = []
while len(pops) > 0:
    for a in pops:
        good = True
        for b in pops:
            if b != a and bigmat[b][a] != 0:
                good = False
        if good:
            order.append(a)
            pops.remove(a)

order.reverse()
print("Computed the matrix of pairings and the upper triangular order.")

Computed the matrix of pairings and the upper triangular order.


In [3]:
def lc(i):
    a = bigmat[i][i]
    d = dict(WCR(a))
    if len(d) != 1:
        print("LC ERROR")
    else:
        return list(d.values())[0]

counter = 0
d_dual = {}
for j in order:
    counter += 1
    print(counter, "out of", len(W), end='\r')
    d_dual[j] = d_sup[j]/lc(j)
    for i in order:
        if  i != j and woke_pairing(d_sub[i], d_dual[j]) != 0:
            d_dual[j] = d_dual[j] - woke_pairing(d_sub[i], d_dual[j])*d_sup[i]/lc(i)
print("Computed the dual basis as the dictionary d_dual.")

KeyboardInterrupt: 

In [16]:
#OPTIONAL CELL: CHECKING THE DUAL BASIS IS VALID.
valid = True
for i in range(len(W)):
    for j in range(len(W)):
        a = woke_pairing(d_sub[i], d_dual[j])
        if a != 0:
            if i != j or a != 1:
                print("ISSUE AT", i, j)
                valid = False
if valid:
    print("Checked dual basis, all is good.")
else:
    print("ISSUES! Dual basis is NOT correct.")

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105


KeyboardInterrupt: 

In [6]:
print("Test")

Test


In [5]:
#Defining [q]^*f_w
q = 23

d_qsub = {}
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_qsub[i] = qfsub(W[i])

In [13]:
#DEPENDS ON THE TYPE! Set the minimal Duflo involution in each left cell. Leave lcell_dict = {} in Type A!

lcell_dict = {}
for a in invols:
    if a not in lcell_dict:
        lcell_dict[a] = a
print("lcell_dict is defined.")

lcell_dict is defined.


In [15]:
def nqpair(w, y):
    #new/normalized q_pairing
    return woke_pairing(d_qsub[list(W).index(w)], d_dual[list(W).index(lcell_dict[y])])

table = []

for w in invols:
    print(str(w))
    print(str(nqpair(w, w)))
    print()
#    table.append([str(w), str(nqpair(w, w))])
#
#for row in table:
#    print('| {:40} | {:1} |'.format(*row))

1
A5(18,18,18,18,18)

s3
A5(18,17,0,17,18) + A5(17,18,0,18,17) + A5(17,18,0,17,19) + A5(19,17,0,18,17) + A5(19,17,0,17,19) + A5(18,18,0,18,18)

s3*s4*s2*s3
-A5(18,17,0,17,18) + A5(17,18,0,18,17) - A5(19,17,0,17,19) + A5(18,18,0,18,18)

s2
A5(17,0,18,18,17) + A5(17,0,18,17,19) + A5(17,0,17,19,18) + A5(18,0,18,18,18)

s2*s3*s2
A5(16,0,0,18,16) + 2*A5(16,0,0,17,18) + A5(16,0,0,16,20) + 2*A5(17,0,0,18,17) + 2*A5(17,0,0,17,19) + A5(18,0,0,18,18)

s2*s3*s1*s2
-A5(17,0,17,19,18) + A5(18,0,18,18,18)

s4
A5(17,18,18,0,17) + A5(19,17,18,0,17) + A5(18,19,17,0,17) + A5(18,18,18,0,18)

s3*s4*s3
A5(16,18,0,0,16) + 2*A5(18,17,0,0,16) + 2*A5(17,18,0,0,17) + A5(20,16,0,0,16) + 2*A5(19,17,0,0,17) + A5(18,18,0,0,18)

s4*s2
A5(17,0,16,0,17) + A5(16,1,16,0,18) + A5(16,0,18,0,16) + A5(16,0,17,1,17) + A5(18,0,16,1,16) + 3*A5(18,0,16,0,18) + A5(17,1,17,0,16) + 2*A5(17,1,16,1,17) + A5(17,1,16,0,19) + 3*A5(17,0,18,0,17) + A5(17,0,17,1,18) + A5(19,0,16,1,17) + A5(19,0,16,0,19) + A5(18,1,17,0,17) + A5(18,0,18,0,1

In [6]:
def normie_nqpair(w, y):
    #new/normalized q_pairing
    return woke_pairing(d_qsub[list(W).index(w)], d_sup[list(W).index(y)])

for w in invols:
    print("normie_dict[" + str(w) + "] = " + str(normie_nqpair(w,w)/lc(list(W).index(w))))

normie_dict[1] = A5(22,22,22,22,22)
normie_dict[s3] = A5(22,21,0,21,22) + A5(21,22,0,22,21) + A5(21,22,0,21,23) + A5(23,21,0,22,21) + A5(23,21,0,21,23) + A5(22,22,0,22,22)
normie_dict[s3*s4*s2*s3] = -A5(22,21,0,21,22) + A5(21,22,0,22,21) - A5(23,21,0,21,23) + A5(22,22,0,22,22)
normie_dict[s2] = A5(21,0,22,22,21) + A5(21,0,22,21,23) + A5(21,0,21,23,22) + A5(22,0,22,22,22)
normie_dict[s2*s3*s2] = A5(20,0,0,22,20) + A5(20,0,0,20,24) + A5(22,0,0,22,22)
normie_dict[s2*s3*s1*s2] = -A5(21,0,21,23,22) + A5(22,0,22,22,22)
normie_dict[s4] = A5(21,22,22,0,21) + A5(23,21,22,0,21) + A5(22,23,21,0,21) + A5(22,22,22,0,22)
normie_dict[s3*s4*s3] = A5(20,22,0,0,20) + A5(24,20,0,0,20) + A5(22,22,0,0,22)
normie_dict[s4*s2] = A5(21,0,20,0,21) + A5(20,1,20,0,22) + A5(20,0,22,0,20) + A5(20,0,21,1,21) + A5(22,0,20,1,20) + A5(21,1,21,0,20) + 2*A5(21,1,20,1,21) + A5(21,1,20,0,23) + A5(21,0,21,1,22) + A5(23,0,20,1,21) + A5(23,0,20,0,23) + A5(22,1,21,0,21) + A5(22,0,22,0,22)
normie_dict[s2*s3*s4*s3*s2] = -A5(21,0